# 02. 탐색적 데이터 분석 (EDA)

## 문제 정의
DRAM 스케일링에서 게이트 산화막(gate oxide)은 **높은 유전율(high-k) + 넓은 밴드갭**을 동시에 만족해야 한다.  
기존 SiO₂(k≈3.9, Eg≈9 eV)를 대체할 후보 소재를 찾는 것이 핵심 과제.  

**이 노트북의 목표:** 15만 개 소재 데이터에서 DRAM high-k 유전체 후보군을 추려내고, 밴드갭 분포와 패턴을 파악한다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import ast

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

df = pd.read_csv('../data/raw_bandgap.csv')
df['elements'] = df['elements'].apply(ast.literal_eval)
print('전체 소재 수:', len(df))
df.head()

## 1. 밴드갭 전체 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 전체 분포
axes[0].hist(df['band_gap'], bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Band Gap (eV)')
axes[0].set_ylabel('Count')
axes[0].set_title('Band Gap Distribution (All)')

# 금속 제외 (band_gap > 0)
df_nonmetal = df[df['band_gap'] > 0.1]
axes[1].hist(df_nonmetal['band_gap'], bins=100, color='darkorange', edgecolor='none')
axes[1].set_xlabel('Band Gap (eV)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Band Gap Distribution (Non-metal, n={len(df_nonmetal)})')

# high-k 유전체 타겟 범위 표시 (Eg > 3 eV)
axes[1].axvline(x=3.0, color='red', linestyle='--', label='Eg = 3 eV (high-k 기준)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/bandgap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'금속(Eg≈0): {len(df[df["band_gap"] <= 0.1]):,}개 ({len(df[df["band_gap"] <= 0.1])/len(df)*100:.1f}%)')
print(f'반도체(0.1~3 eV): {len(df[(df["band_gap"] > 0.1) & (df["band_gap"] <= 3.0)]):,}개')
print(f'wide-gap(>3 eV): {len(df[df["band_gap"] > 3.0]):,}개')

## 2. DRAM high-k 유전체 후보군 필터링

**선별 기준:**
- `is_stable = True` — 열역학적으로 안정한 소재만
- `band_gap > 3.0 eV` — 절연 특성 필요 (누설전류 억제)
- 산화물 계열 — O 포함, 독성/희귀 원소 제외
- `nelements >= 2` — 단원소 제외

In [ ]:
# 독성/방사성/희귀 원소 제외
exclude = {'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No', 'Lr',
           'Po', 'At', 'Rn', 'Fr', 'Ra', 'Tc', 'Pm'}

def contains_oxygen(elements):
    return 'O' in elements

def no_excluded(elements):
    return len(set(elements) & exclude) == 0

df_highk = df[
    (df['is_stable'] == True) &
    (df['band_gap'] > 3.0) &
    (df['nelements'] >= 2) &
    (df['elements'].apply(contains_oxygen)) &
    (df['elements'].apply(no_excluded))
].copy()

print(f'high-k 후보군: {len(df_highk):,}개')
print(df_highk['band_gap'].describe())
df_highk.head(10)

## 3. 알려진 high-k 소재 포함 확인
HfO₂, ZrO₂, Al₂O₃ — 실제 DRAM에 사용 중인 소재들이 데이터에 있는지 확인.

In [ ]:
known_highk = ['HfO2', 'ZrO2', 'Al2O3', 'TiO2', 'La2O3', 'Y2O3', 'HfSiO4', 'ZrSiO4']

print('=== 알려진 high-k 소재 데이터 확인 ===')
for mat in known_highk:
    match = df[df['formula'] == mat]
    if len(match) > 0:
        best = match.loc[match['energy_above_hull'].idxmin()]
        print(f'{mat:12s} | band_gap: {best["band_gap"]:.2f} eV | stable: {best["is_stable"]} | E_hull: {best["energy_above_hull"]:.4f}')
    else:
        print(f'{mat:12s} | 데이터 없음')

## 4. 원소 구성 분석 — 어떤 원소가 많이 등장하나

In [ ]:
from collections import Counter

# high-k 후보군에서 O 제외한 양이온 원소 빈도
cation_list = []
for elems in df_highk['elements']:
    cation_list.extend([e for e in elems if e != 'O'])

cation_count = Counter(cation_list)
top20 = pd.DataFrame(cation_count.most_common(20), columns=['element', 'count'])

plt.figure(figsize=(10, 4))
plt.bar(top20['element'], top20['count'], color='steelblue')
plt.xlabel('Element')
plt.ylabel('Count')
plt.title('Top 20 Cations in high-k Candidates (Eg > 3 eV, stable oxides)')
plt.tight_layout()
plt.savefig('../data/cation_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

print(top20)

## 5. 필터링 데이터 저장

In [ ]:
df_highk.to_csv('../data/highk_candidates.csv', index=False)
print(f'저장 완료: data/highk_candidates.csv ({len(df_highk):,}개)')